In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun
import os

In [13]:
WEATHER_API = os.getenv("WEATHER_API")

In [2]:
search_tool = DuckDuckGoSearchRun()

In [3]:
load_dotenv()

True

In [14]:
@tool
def get_weather_data(city: str) -> str:
    """ 
    This function fetches the current weather data for a given city.
    """

    url = f'http://api.weatherstack.com/current?access_key={WEATHER_API}&query={city}'
    response = requests.get(url)
    return response.json()


In [15]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0
)

In [16]:
from langchain_classic.agents import create_react_agent , AgentExecutor
from langchain_core.prompts import PromptTemplate

In [17]:
## pull the react prompt from langchain hub

prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")  # pulls the standard react agent prompt

In [18]:
agent = create_react_agent(
    llm = llm,
    tools = [search_tool , get_weather_data],
    prompt=prompt
)

In [19]:
agent_executor = AgentExecutor(
    agent=agent,
    tools = [search_tool, get_weather_data],
    verbose=True
)

In [20]:
response = agent_executor.invoke({"input" : "Find the capital of Bhubaneshwar , then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: capital of BhubaneshwarBhubaneshwar, city,capitalofOdisha (Orissa) state, eastern India . ... Between the 5th and 10th centuries ce it was the provincialcapitalof... BhubaneshwarBhubaneshwaris the moderncapitalofOdisha. ...Bhubaneshwaris the moderncapitalofOdisha in independent India. Bhubaneswar is thecapitaland largest cityofthe Indian stateofOdisha, located in the Khordha district. ... Although the modern cityof... Bhubaneshwaris a city bestowed with vibrant art and culture and was once reigned by Kalinga Kingdom, It is now renowned as thecapitalofOrissa. Bhubaneswar is oneofthe most popularcapitalcities in the country also referred to as Temple cityofIndia as rightly called so.I have already found that Bhubaneshwar is the capital of Odisha. Now I need to find its current weather condition.
Action: get_weather_data
Action Input: Bhubaneshwar{'request': {'type': 'City', 'query': 'Bhubaneshwar, India', 'la